# Manuscript:   A reference map of CDK-cyclin dependencies across human cancers
# Project:      Codependencies
# Part:         E. GSEA analysis
# Author:       Agustín Sánchez-Belmonte
# Date:         04/09/23
# Last update:  08/02/26

# GSEA analysis from rnk file

In [1]:
import os
import pandas as pd
import gseapy as gspy
import numpy as np

## 1. Create rnk file

In [ ]:
markers2 = pd.read_excel(
    'Supplementary_tables/TableS5_DE_ ClusterD_vs_rest.xlsx',
    sheet_name='BREAST',  # nombre de la hoja
    index_col=0
)
markers2

,logFC,AveExpr,t,P.Value,adj.P.Val,B,score
Gene,,,,,,,
CCND1,-3.597569,6.016073,-10.667823,3.584504e-19,6.887624e-15,32.901745,-50.948522
KIF5C,3.440706,1.747674,10.070728,9.917681e-18,9.528412e-14,29.694348,44.801367
CDK5R1,1.998057,2.255155,9.773853,5.144342e-17,3.294951e-13,28.103333,24.940052
ELAVL2,2.877268,1.498872,9.185013,1.324621e-15,6.363148e-12,24.962736,32.214838
TMEFF1,1.977232,1.406723,9.127874,1.812642e-15,6.965984e-12,24.659443,22.060010
...,...,...,...,...,...,...,...
TAF11L2,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00,-6.605484,0.000000
SCYGR3,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00,-6.605484,0.000000
SCYGR7,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00,-6.605484,0.000000


In [3]:

# set gen column
markers2['#Gene'] = markers2.index
markers2['#Gene'] = markers2['#Gene'].str.upper()

markers2['Score'] = markers2.logFC*-np.log10(markers2["adj.P.Val"])

# select only score and gene
markers2 = markers2[['#Gene','Score']]
markers2 = markers2.dropna()
markers2 = markers2.sort_values(by='Score', ascending=False)
markers2

,#Gene,Score
CDKN2A,CDKN2A,136.337168
CCNE1,CCNE1,52.363175
CDKN2B,CDKN2B,23.529145
INA,INA,21.761650
RBP7,RBP7,18.272589
...,...,...
MYEOV,MYEOV,-7.211975
RAC2,RAC2,-7.339975
STAT6,STAT6,-8.497839
CCND1,CCND1,-9.767559


## 2.GSEA analysis

In [4]:
def rnk_to_geseapy(pre_res, rnk, gset, outdir, samplename):
    """Run GSEApy and merge index results with a previous table."""
    
    gene_set    = gset + ".gmt"
    pre_res_new = gspy.prerank(rnk=rnk, gene_sets= gene_set, processes=4, permutation_num=10000,
                               outdir=outdir, graph_num=60, format='png', seed=6)
    pre_res_new_df = pd.DataFrame(pre_res_new.res2d.sort_index())
    pre_res_new_df.to_csv(outdir + "/" + "GSEApy_results.tsv", sep="\t")
    pre_res     = pd.concat([pre_res, pre_res_new.res2d.sort_index()])
    n           = pre_res_new.res2d.shape[0]                                  # number of new files added
    pre_res.iloc[-n:, -1] = samplename                                        # add sample code to last column
    return pre_res

def run_all(DESKTOP, dataname, groupA, groupB, pre_res, rnk):
    """RUN all scripts"""
    
    # Organize variables & folders
    sample = groupA + "_vs_" + groupB
    outdir = DESKTOP + "/" + dataname + "_" + sample
    samplename = dataname + "_" + groupA + "_vs_" + groupB

    if not os.path.exists(outdir):
        os.mkdir(outdir)

    pre_res = rnk_to_geseapy(pre_res, rnk, gset, outdir, samplename)
    #pre_res.to_csv(outdir + "/GSEApy_results.tsv", sep="\t")
    #print(pre_res.head(3))
    
    return pre_res

In [5]:
dataname = "PANCANCER"

DESKTOP = '/Users/asanchezb/Desktop/'
rnk_file = markers2

# Select Pathways
GENESETS = "/Users/asanchezb/Desktop/OTHERS/GSEA/"
gset = GENESETS + "h.all.v2024.1.Hs.symbols"

#h.all.v7.2.symbols
#h.c2.cp.v7.2.symbols_mix

# Empty dataframe to accumulate all GSEA results
pre_res = pd.read_csv(GENESETS + "pre_res_empy.tsv", sep="\t", index_col="Term")

# Column and groups

groupA = "D"              # label for score in adata.obs
groupB = "rest"                # label for score in adata.obs
pre_res = run_all(DESKTOP, dataname, groupA, groupB, pre_res, rnk_file)


pre_res.to_csv(DESKTOP + "GSEApy_Combined_results.tsv", sep="\t")